### Check Colab

In [1]:
import sys
print(sys.executable)
!nvidia-smi

/usr/bin/python3
Mon Sep 21 06:33:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------------

### Enter the project folder

In [4]:
%cd /content
!ls scripts
!ls khmer_transliteration

/content
auto_label_training_examples.py  inspect_asr_dataset.py
build_asr_manifest.py		 promote_label_data.py
candidate_test_report.py	 __pycache__
collect_ranking_examples.py	 suggestion_test_report.py
dataset_summary.py		 train_ranking_model.py
duplicate_data_check.py		 validate_asr_manifest.py
__init__.py
asr_text.py		history.py	  paths.py
auto_label_training.py	__init__.py	  __pycache__
candidate_generator.py	label_review.py   ranking_features.py
collection.py		mapping_rules.py  suggestion_engine.py
dictionary_lookup.py	normalizer.py	  web.py


### Install Dependency

In [9]:
%pip install datasets
%pip install -r requirements-asr.txt

### Confirm Requirements File

In [10]:
!ls requirements-asr.txt


requirements-asr.txt


### Generate *1000* Row Manifests

In [12]:
!python scripts/build_asr_manifest.py --target-rows 1000 --scan-rows 5000 --shuffle-buffer 5000 --output-dir data/voice_1000

Resolving data files: 100% 4447/4447 [00:00<00:00, 22908.55it/s]
Resolving data files: 100% 4447/4447 [00:00<00:00, 23928.56it/s]
Scanned rows: 5000
Selected rows: 1000
Split counts: {'train': 793, 'validation': 81, 'test': 126}
Saved manifests to: data/voice_1000


### Check the Generated Files

In [13]:
!ls -lh data/voice_1000

total 944K
-rw-r--r-- 1 root root 3.1K Sep 16 09:35 manifest_summary.json
-rw-r--r-- 1 root root  97K Sep 16 09:35 test.csv
-rw-r--r-- 1 root root 767K Sep 16 09:35 train.csv
-rw-r--r-- 1 root root  72K Sep 16 09:35 validation.csv


### Validate the 1000-Row Manifests

In [14]:
!python scripts/validate_asr_manifest.py --manifest-dir data/voice_1000 --output data/voice_1000/manifest_validation.json

Resolving data files: 100% 4447/4447 [00:00<00:00, 21315.36it/s]
Resolving data files: 100% 4447/4447 [00:00<00:00, 21741.82it/s]
Manifest rows: 1000
Checked rows: 1000
Valid WAV files: 1000
Mismatches: 0
Saved report to: /content/data/voice_1000/manifest_validation.json


In [15]:
import json
from pathlib import Path

validation = json.loads(Path('data/voice_1000/manifest_validation.json').read_text(encoding='utf-8'))
print('Manifest rows:', validation['manifest_rows'])
print('Checked rows:', validation['checked_rows'])
print('Valid WAV files:', validation['valid_wav_count'])
print('Mismatches:', validation['mismatch_count'])
print('Sampling rates:', validation['sampling_rates'])

Manifest rows: 1000
Checked rows: 1000
Valid WAV files: 1000
Mismatches: 0
Sampling rates: {'16000': 972, '48000': 28}


### Save the Manifests to Google Drive

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
from pathlib import Path
import shutil

source = Path('data/voice_1000')
destination = Path('/content/drive/MyDrive/khmer_asr/voice_1000')
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(source, destination, dirs_exist_ok=True)
print(f'Saved manifests to {destination}')

Saved manifests to /content/drive/MyDrive/khmer_asr/voice_1000
